# 🏏 Cricket Batter Pose Estimation & Bat Segmentation (ViTPose+ & YOLOv11-seg)

This notebook runs a two-pass computer vision pipeline to track cricket batting mechanics and segment the bat blade without temporal jitter or limb-swapping artifacts:

1. **Why ViTPose+ over Google MediaPipe?**
   * We initially experimented with **Google MediaPipe (BlazePose)**, which struggled with side-on batting stances, thick batting pads/gloves, and motion blur—causing frequent left/right leg swaps and floating ankle joints.
   * Upgrading to **RT-DETR (`PekingU/rtdetr_r50vd_coco_o365`) + ViTPose+ Large (`usyd-community/vitpose-plus-large`)** on GPU accurately resolves occluded joints even through heavy protective gear.
2. **Offline Temporal Cleaning:** Fixes left/right limb swaps across full kinematic chains, prevents knee/ankle collapse on batting pads, interpolates missing frames, and applies zero-lag Gaussian smoothing.
3. **Wrist-Guided Bat Segmentation:** Uses **YOLOv11m-seg** at 1280px resolution with a forearm-pivot rotation hold so the bat mask stays locked to the blade during fast swings.

## 1. Install Dependencies
Run the cell below to upgrade `transformers` (required for `VitPoseForPoseEstimation` and `RTDetrForObjectDetection`) and install `ultralytics` for YOLOv11 segmentation. *(Make sure your Kaggle Accelerator is set to **GPU T4 x2** or **GPU P100**).*

In [1]:
!pip install -q transformers ultralytics

In [2]:
VIDEO_IN  = "/kaggle/input/test_swing/test_swing.mp4"
VIDEO_OUT = "/kaggle/working/swing_pose_bat_free.mp4"

## 2. Full 2-Pass Pose & Bat Segmentation Pipeline
* **Pass 1 (Inference):** Detects the batter inside `BATTER_ROI`, extracts 17 COCO keypoints via ViTPose+ Large, and segments the cricket bat polygon near the batter's wrists.
* **Offline Post-Processing:** Clamps shadow outliers, prevents pad collapse, fixes L/R limb swaps, interpolates short occlusion gaps, and rotates held bat masks with the forearms.
* **Pass 2 (Rendering):** Draws sleek anti-aliased skeleton lines (`thickness=3`), alpha-blends the orange bat mask, and encodes an H.264 `.mp4` via FFmpeg.

In [8]:
import math, subprocess, warnings
import cv2
import numpy as np
import torch
from transformers import AutoProcessor, RTDetrForObjectDetection, VitPoseForPoseEstimation
from ultralytics import YOLO

warnings.filterwarnings("ignore")

# ===================== SETTINGS =====================
VIDEO_IN   = "/kaggle/input/datasets/malikahmadrasheed223/test-swing/test_swing.mp4"
RAW_OUT    = "/kaggle/working/raw.mp4"
VIDEO_OUT  = "/kaggle/working/batter_final_without_ball_track.mp4"

BATTER_ROI = (0.28, 0.00, 0.78, 1.00)   # batter zone (left, top, right, bottom)
SHOW_ROI   = False

# --- pose ---
POSE_MODEL = "usyd-community/vitpose-plus-large"
THRESH     = 0.35       # keypoint confidence cutoff (raised slightly to reject shadow false positives)
OUT_PX     = 0.035      # glitch threshold (fraction of width)
GAP_MAX    = 15         # fill missing joints for gaps up to this many frames
SIGMA      = 1.15       # temporal smoothing (frames)

# --- bat ---
SEGMENT_BAT = True
YOLO_MODEL  = "yolo11m-seg.pt"
YOLO_IMGSZ  = 1280
BAT_CONF    = 0.04      # low threshold so motion-blurred cricket bats aren't dropped
BAT_HOLD    = 4         # frames to carry & rotate the bat when YOLO misses
BAT_COLOR   = (0, 140, 255)

# --- ball trail ---
TRACK_BALL    = False
BALL_CONF     = 0.03
USE_MOTION    = True    # detect fast moving ball blobs via frame differencing + MOG2
MIN_PTS       = 3       # allow fast cricket deliveries (often only 3-5 frames in view)
MAX_STEP      = 0.18    # max ball move per frame (fraction of width)
CONTACT_DIST  = 0.36    # distance (fraction of height) to bat/hands to count as hit
SWING_WIN_SEC = 0.85    # window around peak bat/hand speed
DEBUG_BALL    = False   # True = draw every filtered ball candidate as a small red dot
PRE_COLOR     = (255, 255, 255)
POST_COLOR    = (60, 255, 60)
# ====================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

det_name = "PekingU/rtdetr_r50vd_coco_o365"
det_proc = AutoProcessor.from_pretrained(det_name)
detector = RTDetrForObjectDetection.from_pretrained(det_name).to(device).eval()

pose_proc = AutoProcessor.from_pretrained(POSE_MODEL)
pose_model = VitPoseForPoseEstimation.from_pretrained(POSE_MODEL).to(device).eval()

yolo = YOLO(YOLO_MODEL) if (SEGMENT_BAT or TRACK_BALL) else None

PURPLE, TEAL = (200, 60, 170), (190, 190, 0)
RED, MAGENTA, WHITE = (80, 80, 255), (220, 60, 255), (255, 255, 255)
SKELETON = [
    ((5, 7), PURPLE), ((7, 9), PURPLE), ((6, 8), TEAL), ((8, 10), TEAL),
    ((11, 13), RED), ((13, 15), RED), ((12, 14), MAGENTA), ((14, 16), MAGENTA),
    ((5, 6), WHITE), ((11, 12), WHITE), ((5, 11), WHITE), ((6, 12), WHITE),
]
LIMBS = [(5, 7), (7, 9), (6, 8), (8, 10), (11, 13), (13, 15), (12, 14), (14, 16)]
PAIRS = [(5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16)]


def ip(p):
    return (int(round(p[0])), int(round(p[1])))


# =====================================================
# PASS 1: run the models on every frame, store results
# =====================================================
def pick_batter(boxes_xyxy, w, h):
    rx1, ry1, rx2, ry2 = BATTER_ROI
    best, best_area = None, 0
    for i, b in enumerate(boxes_xyxy):
        cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
        if rx1 * w <= cx <= rx2 * w and ry1 * h <= cy <= ry2 * h:
            area = (b[2] - b[0]) * (b[3] - b[1])
            if area > best_area:
                best, best_area = i, area
    return best


prev_box = None

@torch.no_grad()
def get_pose(frame_bgr):
    global prev_box
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    inp = det_proc(images=rgb, return_tensors="pt").to(device)
    res = det_proc.post_process_object_detection(
        detector(**inp), target_sizes=torch.tensor([(h, w)]), threshold=0.45)[0]
    boxes = res["boxes"][res["labels"] == 0].cpu().numpy()
    idx = pick_batter(boxes, w, h) if len(boxes) else None
    if idx is None:
        if prev_box is None:
            return None, None
        b = prev_box.copy()
    else:
        b = boxes[idx].astype(float)
        # Light smoothing (0.85 current / 0.15 prev) so box never lags behind fast lunges
        if prev_box is not None:
            b = 0.85 * b + 0.15 * prev_box
        prev_box = b.copy()

    bw, bh = b[2] - b[0], b[3] - b[1]
    # Generous padding so knees/ankles/wrists are never clipped at box edges
    x1, y1 = max(0, b[0] - 0.18 * bw), max(0, b[1] - 0.12 * bh)
    x2, y2 = min(w - 1, b[2] + 0.18 * bw), min(h - 1, b[3] + 0.12 * bh)
    box = np.array([[x1, y1, x2 - x1, y2 - y1]])   # xywh
    pin = pose_proc(rgb, boxes=[box], return_tensors="pt").to(device)
    pout = pose_model(**pin, dataset_index=torch.tensor([0], device=device))
    p = pose_proc.post_process_pose_estimation(pout, boxes=[box])[0][0]
    return (
        p["keypoints"].cpu().numpy().astype(float),
        p["scores"].cpu().numpy().astype(float),
        (x1, y1, x2, y2),
    )


def run_yolo(frame):
    # classes: 32 = sports ball, 34 = baseball bat, 38 = tennis racket (often detects wide cricket blades)
    r = yolo(frame, imgsz=YOLO_IMGSZ, conf=min(BAT_CONF, BALL_CONF),
             classes=[32, 34, 38], retina_masks=True, verbose=False)[0]
    bats, balls = [], []
    if r.boxes is None or len(r.boxes) == 0:
        return bats, balls
    cls = r.boxes.cls.cpu().numpy().astype(int)
    conf = r.boxes.conf.cpu().numpy()
    xyxy = r.boxes.xyxy.cpu().numpy()
    polys = r.masks.xy if r.masks is not None else [None] * len(cls)
    for c, cf, bb, pl in zip(cls, conf, xyxy, polys):
        if c in (34, 38) and cf >= BAT_CONF and pl is not None and len(pl) >= 3:
            # Boost class 34 slightly over 38
            adj_cf = float(cf) if c == 34 else float(cf) * 0.85
            bats.append((adj_cf, bb, pl.astype(np.int32)))
        elif c == 32 and cf >= BALL_CONF:
            bw, bh = bb[2] - bb[0], bb[3] - bb[1]
            if max(bw, bh) <= 0.055 * W and max(bw, bh) / max(1.0, min(bw, bh)) < 2.8:
                balls.append(((bb[0] + bb[2]) / 2, (bb[1] + bb[3]) / 2, float(cf)))
    return bats, balls


def choose_bat(bats, kp, sc, frame_bgr):
    rx1, ry1, rx2, ry2 = BATTER_ROI
    pts = [kp[i] for i in (9, 10) if sc[i] > THRESH and not np.isnan(kp[i]).any()]
    wrist = np.mean(pts, axis=0) if pts else None

    best, best_key = None, None
    for cf, bb, poly in bats:
        x1, y1, x2, y2 = bb
        if x2 < rx1 * W or x1 > rx2 * W or y2 < ry1 * H or y1 > ry2 * H:
            continue
        # Reject tiny blobs or huge net/pad regions
        area = cv2.contourArea(poly)
        if area < 80 or area > 0.04 * W * H:
            continue
        rect = cv2.minAreaRect(poly)
        rw, rh = rect[1]
        if min(rw, rh) < 3 or max(rw, rh) / max(1.0, min(rw, rh)) < 1.6:
            continue

        dist = 0.0
        if wrist is not None:
            # Distance from wrist to closest point on polygon
            d_poly = -cv2.pointPolygonTest(poly.reshape(-1, 1, 2).astype(np.float32),
                                           (float(wrist[0]), float(wrist[1])), True)
            dist = max(0.0, d_poly)
            if dist > 0.22 * H:   # bat grip must be near the wrists
                continue
        key = (dist - 25.0 * cf)
        if best_key is None or key < best_key:
            best, best_key = poly, key

    return best


bgs = cv2.createBackgroundSubtractorMOG2(history=120, varThreshold=32, detectShadows=False)
_k3 = np.ones((3, 3), np.uint8)
prev_gray = None

def motion_blobs(frame):
    global prev_gray
    fg1 = bgs.apply(frame)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if prev_gray is not None:
        diff = cv2.absdiff(gray, prev_gray)
        _, fg2 = cv2.threshold(diff, 22, 255, cv2.THRESH_BINARY)
        fg = cv2.bitwise_or(fg1, fg2)
    else:
        fg = fg1
    prev_gray = gray

    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, _k3)
    cnts, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    s = (W / 1280.0) ** 2
    out = []
    for c in cnts:
        a = cv2.contourArea(c)
        if a < 4 * s or a > 750 * s:
            continue
        x, y, bw, bh = cv2.boundingRect(c)
        if max(bw, bh) > 0.05 * W or max(bw, bh) / max(1, min(bw, bh)) > 4.2:
            continue
        out.append((x + bw / 2, y + bh / 2, 0.25))
    return [] if len(out) > 85 else out


cap = cv2.VideoCapture(VIDEO_IN)
if not cap.isOpened():
    raise SystemExit(f"Could not open {VIDEO_IN}")
fps = cap.get(cv2.CAP_PROP_FPS) or 30
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

KP_list, SC_list, BOX_list, BAT_raw, CAND = [], [], [], [], []
n = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    pose = get_pose(frame)
    if pose[0] is None:
        kp, sc, pbox = np.full((17, 2), np.nan), np.zeros(17), None
    else:
        kp, sc, pbox = pose
    KP_list.append(kp)
    SC_list.append(sc)
    BOX_list.append(pbox)

    bats, balls = run_yolo(frame) if yolo is not None else ([], [])
    BAT_raw.append(choose_bat(bats, kp, sc, frame) if SEGMENT_BAT else None)

    cands = list(balls) if TRACK_BALL else []
    if TRACK_BALL and USE_MOTION:
        cands += motion_blobs(frame)
    CAND.append(cands)

    n += 1
    if n % 20 == 0:
        print(f"pass 1: {n}/{total}", end="\r")
cap.release()
N = len(KP_list)
if N == 0:
    raise SystemExit("No frames read.")
print(f"\npass 1 done: {N} frames")


# =====================================================
# CLEAN THE POSE OFFLINE (stops leg swaps & shadow jumps)
# =====================================================
def clamp_to_person_and_legs(KP, BOX_list):
    """Reject lower-body keypoints that jump into the background/shadow behind the batter."""
    for t in range(len(KP)):
        kp = KP[t]
        if np.isnan(kp).all():
            continue
        # Ensure knees & ankles stay plausibly underneath the hips
        hips = kp[[11, 12]]
        hok = ~np.isnan(hips).any(axis=1)
        if hok.any():
            hx, hy = hips[hok].mean(axis=0)
            for j in (13, 14, 15, 16):
                if not np.isnan(kp[j]).any():
                    # Knee/ankle cannot be above the hip or wildly sideways
                    if kp[j, 1] < hy - 0.03 * H or abs(kp[j, 0] - hx) > 0.18 * W:
                        kp[j] = np.nan


def fix_swaps(KP):
    """Enforce temporal left/right consistency across full lower-body and upper-body chains."""
    LOWER_L, LOWER_R = [11, 13, 15], [12, 14, 16]
    UPPER_L, UPPER_R = [5, 7, 9], [6, 8, 10]

    for chain_l, chain_r in [(LOWER_L, LOWER_R), (UPPER_L, UPPER_R)]:
        prev_l, prev_r = None, None
        for t in range(len(KP)):
            cur_l = KP[t, chain_l]
            cur_r = KP[t, chain_r]
            valid = (~np.isnan(cur_l).any(axis=1)) & (~np.isnan(cur_r).any(axis=1))
            if valid.sum() >= 2 and prev_l is not None:
                v_prev = (~np.isnan(prev_l).any(axis=1)) & (~np.isnan(prev_r).any(axis=1)) & valid
                if v_prev.sum() >= 2:
                    d_keep = np.linalg.norm(cur_l[v_prev] - prev_l[v_prev], axis=1).sum() + \
                             np.linalg.norm(cur_r[v_prev] - prev_r[v_prev], axis=1).sum()
                    d_swap = np.linalg.norm(cur_l[v_prev] - prev_r[v_prev], axis=1).sum() + \
                             np.linalg.norm(cur_r[v_prev] - prev_l[v_prev], axis=1).sum()
                    if d_swap + 4.0 < d_keep:
                        KP[t, chain_l], KP[t, chain_r] = KP[t, chain_r].copy(), KP[t, chain_l].copy()
            # Update anchor with valid points
            if (~np.isnan(KP[t, chain_l]).any(axis=1)).sum() >= 2 and \
               (~np.isnan(KP[t, chain_r]).any(axis=1)).sum() >= 2:
                prev_l = KP[t, chain_l].copy()
                prev_r = KP[t, chain_r].copy()


def prevent_leg_collapse(KP):
    """If both knees or both ankles collapse onto the exact same pixel (detector double-counting one leg),
    drop the lower-confidence one so fill_gaps interpolates it cleanly."""
    for t in range(len(KP)):
        for a, b in [(11, 12), (13, 14), (15, 16)]:
            if not np.isnan(KP[t, a]).any() and not np.isnan(KP[t, b]).any():
                if np.linalg.norm(KP[t, a] - KP[t, b]) < 0.008 * W:
                    if SC[t, a] < SC[t, b]:
                        KP[t, a] = np.nan
                    else:
                        KP[t, b] = np.nan


def drop_long_limbs(KP, factor=1.45):
    for a, b in LIMBS:
        d = np.linalg.norm(KP[:, a] - KP[:, b], axis=1)
        med = np.nanmedian(d)
        if np.isfinite(med):
            KP[d > factor * med, b] = np.nan


def drop_spikes(KP, k, thr_px):
    half, out = k // 2, KP.copy()
    for j in range(KP.shape[1]):
        for t in range(len(KP)):
            if np.isnan(KP[t, j]).any():
                continue
            win = KP[max(0, t - half):min(len(KP), t + half + 1), j]
            win = win[~np.isnan(win).any(axis=1)]
            if len(win) < 3:
                continue
            if np.linalg.norm(KP[t, j] - np.median(win, axis=0)) > thr_px:
                out[t, j] = np.nan
    return out


def fill_gaps(KP, max_gap):
    Nn, out = len(KP), KP.copy()
    for j in range(KP.shape[1]):
        valid = ~np.isnan(KP[:, j]).any(axis=1)
        t = 0
        while t < Nn:
            if valid[t]:
                t += 1
                continue
            s = t
            while t < Nn and not valid[t]:
                t += 1
            e = t
            if s > 0 and e < Nn and (e - s) <= max_gap:
                for ax in range(2):
                    out[s:e, j, ax] = np.interp(np.arange(s, e), [s - 1, e], [KP[s - 1, j, ax], KP[e, j, ax]])
    return out


def smooth(KP, sigma):
    r = int(3 * sigma) + 1
    k = np.exp(-0.5 * (np.arange(-r, r + 1) / sigma) ** 2)
    out = KP.copy()
    for j in range(KP.shape[1]):
        valid = ~np.isnan(KP[:, j]).any(axis=1)
        wsum = np.convolve(valid.astype(float), k, mode="same")
        for ax in range(2):
            x = np.where(valid, KP[:, j, ax], 0.0)
            num = np.convolve(x, k, mode="same")
            out[:, j, ax] = np.where(valid & (wsum > 1e-6), num / np.maximum(wsum, 1e-6), np.nan)
    return out


KP, SC = np.array(KP_list), np.array(SC_list)
KP[SC < THRESH] = np.nan
clamp_to_person_and_legs(KP, BOX_list)
prevent_leg_collapse(KP)
fix_swaps(KP)
drop_long_limbs(KP)
KP = drop_spikes(KP, 7, OUT_PX * W)
KP = fill_gaps(KP, GAP_MAX)
fix_swaps(KP)
KP = smooth(KP, SIGMA)


def hands_of(kp):
    pts = kp[[9, 10]]
    ok = ~np.isnan(pts).any(axis=1)
    return pts[ok].mean(axis=0) if ok.any() else np.array([np.nan, np.nan])


def forearms_angle(kp):
    """Estimate forearm direction vector from elbows (7,8) to wrists (9,10) to rotate held bat."""
    vecs = []
    for e, w in [(7, 9), (8, 10)]:
        if not np.isnan(kp[e]).any() and not np.isnan(kp[w]).any():
            vecs.append(kp[w] - kp[e])
    if not vecs:
        return None
    v = np.mean(vecs, axis=0)
    return math.atan2(v[1], v[0])


def fill_series(a):
    a = a.copy()
    for ax in range(2):
        v = ~np.isnan(a[:, ax])
        if v.sum() >= 2:
            a[:, ax] = np.interp(np.arange(len(a)), np.where(v)[0], a[v, ax])
    return a


hands = np.array([hands_of(KP[t]) for t in range(N)])
hands_f = fill_series(hands)
angles = [forearms_angle(KP[t]) for t in range(N)]


def transform_poly(poly, pivot_old, pivot_new, d_angle):
    """Shift and rotate bat polygon around the wrists so held bats don't lag below the real bat."""
    pts = poly.astype(np.float64) - pivot_old
    c, s = math.cos(d_angle), math.sin(d_angle)
    R = np.array([[c, -s], [s, c]])
    pts = pts @ R.T + pivot_new
    return np.round(pts).astype(np.int32)


# bat: carry & interpolate/rotate the mask across short missed gaps
BATF = [None] * N
valid_bats = [t for t in range(N) if BAT_raw[t] is not None]
for t in range(N):
    if BAT_raw[t] is not None:
        BATF[t] = BAT_raw[t]

# Forward/backward short-gap hold with wrist translation + forearm rotation
last_poly, last_t = None, -999
for t in range(N):
    if BATF[t] is not None:
        last_poly, last_t = BATF[t], t
    elif last_poly is not None and (t - last_t) <= BAT_HOLD:
        if not np.isnan(hands_f[t]).any() and not np.isnan(hands_f[last_t]).any():
            da = 0.0
            if angles[t] is not None and angles[last_t] is not None:
                da = angles[t] - angles[last_t]
                # wrap to [-pi, pi] and damp slightly
                da = ((da + math.pi) % (2 * math.pi) - math.pi) * 0.7
            BATF[t] = transform_poly(last_poly, hands_f[last_t], hands_f[t], da)


# =====================================================
# BALL: filter candidates -> link into tracks -> find the hit
# =====================================================
def body_mask(kp):
    m = np.zeros((H, W), np.uint8)
    ok = lambda *ids: not np.isnan(kp[list(ids)]).any()
    if ok(5, 6, 12, 11):
        cv2.fillPoly(m, [kp[[5, 6, 12, 11]].astype(np.int32)], 255)
    for a, b in [(11, 13), (13, 15), (12, 14), (14, 16), (5, 7), (7, 9), (6, 8), (8, 10)]:
        if ok(a, b):
            cv2.line(m, ip(kp[a]), ip(kp[b]), 255, max(3, int(0.06 * H)))
    face = kp[:5]
    fok = ~np.isnan(face).any(axis=1)
    if fok.any():
        c = face[fok].mean(axis=0)
        cv2.circle(m, ip(c), int(0.065 * H), 255, -1)
    return m


def ref_dist(t, x, y):
    ds = []
    h_ = hands_f[t]
    if not np.isnan(h_).any():
        ds.append(math.hypot(x - h_[0], y - h_[1]))
    poly = BATF[t]
    if poly is not None:
        d = cv2.pointPolygonTest(poly.reshape(-1, 1, 2).astype(np.float32), (float(x), float(y)), True)
        ds.append(max(0.0, -d))
    return min(ds) if ds else None


def dense(points):
    d = {p[0]: (p[1], p[2]) for p in points}
    ts = sorted(d)
    if len(ts) < 2:
        return []
    T = np.arange(ts[0], ts[-1] + 1)
    X = np.interp(T, ts, [d[t][0] for t in ts])
    Y = np.interp(T, ts, [d[t][1] for t in ts])
    if len(T) >= 5:
        k = np.ones(3) / 3
        X = np.convolve(np.pad(X, (1, 1), mode="edge"), k, mode="valid")
        Y = np.convolve(np.pad(Y, (1, 1), mode="edge"), k, mode="valid")
    return list(zip(T.tolist(), X.tolist(), Y.tolist()))


pre_path, post_path, hit_t, hit_xy = [], [], None, None
cands_f = [[] for _ in range(N)]

if TRACK_BALL:
    # 1) remove candidates on the body, hands and bat
    for t in range(N):
        kp = KP[t]
        body = None if np.isnan(kp).all() else body_mask(kp)
        wr = [kp[i] for i in (9, 10) if not np.isnan(kp[i]).any()]
        for (x, y, cf) in CAND[t]:
            xi, yi = int(x), int(y)
            if not (0 <= xi < W and 0 <= yi < H):
                continue
            if body is not None and body[yi, xi]:
                continue
            if any(math.hypot(x - p[0], y - p[1]) < 0.04 * H for p in wr):
                continue
            poly = BATF[t]
            if poly is not None and cv2.pointPolygonTest(
                    poly.reshape(-1, 1, 2).astype(np.float32), (float(x), float(y)), True) > -0.015 * H:
                continue
            cands_f[t].append((x, y, cf))

    # 2) remove static objects (rubber duck, stumps, background logos)
    CELL = max(8, int(0.018 * W))
    counts = {}
    for c in cands_f:
        seen = set()
        for x, y, _ in c:
            key = (int(x // CELL), int(y // CELL))
            if key not in seen:
                seen.add(key)
                counts[key] = counts.get(key, 0) + 1
    STATIC_N = max(8, int(0.07 * N))

    def is_static(x, y):
        kx, ky = int(x // CELL), int(y // CELL)
        return sum(counts.get((kx + i, ky + j), 0) for i in (-1, 0, 1) for j in (-1, 0, 1)) >= STATIC_N

    cands_f = [[c for c in cs if not is_static(c[0], c[1])] for cs in cands_f]

    # 3) velocity-predicted track linking (handles fast ball motion & missed frames)
    MAX_GAP = max(5, int(0.35 * fps))
    tracks, active = [], []
    for t in range(N):
        cs = cands_f[t]
        pairs = []
        for ti, tr in enumerate(active):
            lt, lx, ly, _ = tr[-1]
            dt = t - lt
            # Predict position using velocity if track has >= 2 points
            if len(tr) >= 2:
                pt0, px0, py0, _ = tr[-2]
                dt_prev = max(1, lt - pt0)
                vx, vy = (lx - px0) / dt_prev, (ly - py0) / dt_prev
                pred_x, pred_y = lx + vx * dt, ly + vy * dt
            else:
                pred_x, pred_y = lx, ly

            gate = MAX_STEP * W * (1.0 + 0.65 * (dt - 1))
            for ci, (x, y, cf) in enumerate(cs):
                d_pred = math.hypot(x - pred_x, y - pred_y)
                d_raw = math.hypot(x - lx, y - ly)
                d = min(d_pred, d_raw)
                if d <= gate and d_raw <= MAX_STEP * W * dt:
                    pairs.append((d - 15.0 * cf, ti, ci))
        pairs.sort()
        used_t, used_c = set(), set()
        for _, ti, ci in pairs:
            if ti in used_t or ci in used_c:
                continue
            used_t.add(ti)
            used_c.add(ci)
            active[ti].append((t, cs[ci][0], cs[ci][1], cs[ci][2]))
        for ci, c in enumerate(cs):
            if ci not in used_c:
                active.append([(t, c[0], c[1], c[2])])
        still = []
        for tr in active:
            (tracks if t - tr[-1][0] > MAX_GAP else still).append(tr)
        active = still
    tracks += active

    good = []
    for tr in tracks:
        if len(tr) < MIN_PTS:
            continue
        plen = sum(math.hypot(tr[i + 1][1] - tr[i][1], tr[i + 1][2] - tr[i][2]) for i in range(len(tr) - 1))
        net_disp = math.hypot(tr[-1][1] - tr[0][1], tr[-1][2] - tr[0][2])
        span = max(1, tr[-1][0] - tr[0][0])
        # Must move meaningfully across the frame and not just jitter in place
        if plen < 0.07 * W or net_disp < 0.05 * W or plen / span < 0.005 * W:
            continue
        good.append(tr)

    print(f"ball: {sum(len(c) for c in cands_f)} candidates kept, {len(tracks)} tracks, {len(good)} good")

    # 4) find the hit: ball track passing closest to bat/hands around the swing
    sp = np.linalg.norm(np.diff(hands_f, axis=0), axis=1) if N > 3 else np.zeros(1)
    sp = np.convolve(sp, np.ones(3) / 3, mode="same")
    t_swing = int(np.argmax(sp)) + 1
    win = int(SWING_WIN_SEC * fps)

    best = None
    for ti, tr in enumerate(good):
        for pi_, (t, x, y, cf) in enumerate(tr):
            d = ref_dist(t, x, y)
            if d is None:
                continue
            # Prefer points inside the swing window, but allow outside with a small penalty
            time_pen = 0.0 if abs(t - t_swing) <= win else 0.15 * H
            if d <= CONTACT_DIST * H:
                score = d + time_pen
                if best is None or score < best[0]:
                    best = (score, ti, pi_)

    # Fallback: if no single point was within CONTACT_DIST, pick the track that gets closest to the bat/hands
    if best is None and len(good) > 0:
        for ti, tr in enumerate(good):
            for pi_, (t, x, y, cf) in enumerate(tr):
                d = ref_dist(t, x, y)
                if d is not None and (best is None or d < best[0]):
                    best = (d, ti, pi_)

    if best is None:
        print("ball: no valid ball tracks found. Set DEBUG_BALL = True to inspect raw detections.")
    else:
        _, ti, pi_ = best
        tr = good[ti]
        hit_pt = tr[pi_]
        hit_t, hit_xy = hit_pt[0], (hit_pt[1], hit_pt[2])
        pre, post = tr[:pi_ + 1], tr[pi_:]

        # If ball track broke before contact, prepend an incoming track that ends near hit_pt
        if len(pre) < 3:
            best_pre = None
            for oi, other in enumerate(good):
                if oi == ti:
                    continue
                l_pt = other[-1]
                if 0 <= hit_t - l_pt[0] <= 15 and math.hypot(l_pt[1] - hit_xy[0], l_pt[2] - hit_xy[1]) <= 0.35 * W:
                    if best_pre is None or len(other) > len(best_pre):
                        best_pre = other
            if best_pre is not None:
                pre = best_pre + [hit_pt]

        # If ball track broke at contact, join an outgoing track that starts right after
        if len(post) < 3:
            best_post = None
            for oi, other in enumerate(good):
                if oi == ti:
                    continue
                f = other[0]
                if -1 <= f[0] - hit_t <= 15 and math.hypot(f[1] - hit_xy[0], f[2] - hit_xy[1]) <= 0.35 * W:
                    if best_post is None or len(other) > len(best_post):
                        best_post = other
            if best_post is not None:
                post = [hit_pt] + best_post

        pre_path, post_path = dense(pre), dense(post)
        print(f"ball: hit at frame {hit_t}, incoming {len(pre_path)} frames, outgoing {len(post_path)} frames")


# =====================================================
# PASS 2: draw everything
# =====================================================
def draw_pose(frame, kp):
    for (i, j), color in SKELETON:
        if not (np.isnan(kp[i]).any() or np.isnan(kp[j]).any()):
            cv2.line(frame, ip(kp[i]), ip(kp[j]), color, 3, cv2.LINE_AA)
    for i in range(5, 17):
        if not np.isnan(kp[i]).any():
            p = ip(kp[i])
            cv2.circle(frame, p, 5, (0, 0, 0), -1, cv2.LINE_AA)
            cv2.circle(frame, p, 3, WHITE, -1, cv2.LINE_AA)


def draw_bat(frame, poly):
    overlay = frame.copy()
    cv2.fillPoly(overlay, [poly], BAT_COLOR)
    frame[:] = cv2.addWeighted(overlay, 0.45, frame, 0.55, 0)
    cv2.polylines(frame, [poly], True, BAT_COLOR, 2, cv2.LINE_AA)


def draw_trail(frame, path, t, color, thick):
    pts = [(int(x), int(y)) for (tt, x, y) in path if tt <= t]
    if len(pts) >= 2:
        arr = np.array(pts, np.int32)
        cv2.polylines(frame, [arr], False, (0, 0, 0), thick + 4, cv2.LINE_AA)
        cv2.polylines(frame, [arr], False, color, thick, cv2.LINE_AA)


ball_at = {tt: (x, y) for (tt, x, y) in pre_path + post_path}

cap = cv2.VideoCapture(VIDEO_IN)
out = cv2.VideoWriter(RAW_OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
for t in range(N):
    ok, frame = cap.read()
    if not ok:
        break
    if BATF[t] is not None:
        draw_bat(frame, BATF[t])
    draw_pose(frame, KP[t])

    if DEBUG_BALL:
        for (x, y, cf) in cands_f[t]:
            cv2.circle(frame, (int(x), int(y)), 4, (0, 0, 255), -1)

    if hit_t is not None:
        draw_trail(frame, pre_path, t, PRE_COLOR, 3)
        draw_trail(frame, post_path, t, POST_COLOR, 5)
        if t in ball_at:
            p = (int(ball_at[t][0]), int(ball_at[t][1]))
            cv2.circle(frame, p, 9, (0, 0, 0), -1, cv2.LINE_AA)
            cv2.circle(frame, p, 7, WHITE, -1, cv2.LINE_AA)
        if t >= hit_t:
            hp = (int(hit_xy[0]), int(hit_xy[1]))
            cv2.circle(frame, hp, 14, (0, 255, 255), 2, cv2.LINE_AA)
            if t - hit_t < int(1.5 * fps):
                cv2.rectangle(frame, (hp[0] + 18, hp[1] - 34), (hp[0] + 78, hp[1] - 8), (0, 0, 0), -1)
                cv2.putText(frame, "HIT", (hp[0] + 24, hp[1] - 14), cv2.FONT_HERSHEY_SIMPLEX,
                            0.7, (0, 255, 255), 2, cv2.LINE_AA)

    if SHOW_ROI:
        cv2.rectangle(frame, (int(BATTER_ROI[0] * W), int(BATTER_ROI[1] * H)),
                      (int(BATTER_ROI[2] * W) - 1, int(BATTER_ROI[3] * H) - 1), (0, 255, 255), 2)
    out.write(frame)
    if t % 20 == 0:
        print(f"pass 2: {t}/{N}", end="\r")

cap.release()
out.release()
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", RAW_OUT,
                "-vcodec", "libx264", "-pix_fmt", "yuv420p", VIDEO_OUT])
print("\nDone:", VIDEO_OUT)

try:
    from IPython.display import FileLink, display
    display(FileLink(VIDEO_OUT))
except Exception:
    pass

Device: cuda


Loading weights:   0%|          | 0/764 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/691 [00:00<?, ?it/s]

pass 1: 620/639
pass 1 done: 639 frames
pass 2: 620/639
Done: /kaggle/working/batter_final_without_ball_track.mp4


/kaggle/working/batter_final_without_ball_track.mp4

In [6]:
import os
for f in os.listdir("/kaggle/working"):
    print(f, round(os.path.getsize(os.path.join("/kaggle/working", f)) / 1e6, 1), "MB")

raw.mp4 11.3 MB
.virtual_documents 0.0 MB
yolo11m-seg.pt 45.4 MB
batter_pose_bat.mp4 3.2 MB
